### Primer chatbot con Gradio

`gr.ChatInterface` arma una UI de chat completa (input, historial, botones) con una sola función: vos escribís la lógica que recibe `message` + `history` y devuelve la respuesta, Gradio se encarga de dibujar todo. Acá lo conectamos a Ollama local — mismo patrón `messages=[...]` de la semana 1, ahora con interfaz real en vez de `print()`.

In [1]:
import gradio as gr
import ollama

def normalizar(content):
    if isinstance(content, list):
        return "".join(p.get("text", "") for p in content if isinstance(p, dict))
    return content

def chat(message, history):
    history_normalizado = [
        {"role": h["role"], "content": normalizar(h["content"])} for h in history
    ]
    messages = history_normalizado + [{"role": "user", "content": message}]
    response = ollama.chat(model="llama3.2", messages=messages)
    return response["message"]["content"]

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Streaming: respuesta token a token

Con `return` esperás a que el modelo termine TODA la respuesta antes de mostrar algo — con prompts largos se siente colgado. Con `yield` en vez de `return`, la función se vuelve generador: cada vez que Ollama entrega un pedazo nuevo (`stream=True`), lo vas acumulando y emitiendo — Gradio actualiza la pantalla en vivo, como el efecto "escribiendo..." de ChatGPT.

In [2]:
import gradio as gr
import ollama

def normalizar(content):
    if isinstance(content, list):
        return "".join(p.get("text", "") for p in content if isinstance(p, dict))
    return content

def chat_streaming(message, history):
    history_normalizado = [
        {"role": h["role"], "content": normalizar(h["content"])} for h in history
    ]
    messages = history_normalizado + [{"role": "user", "content": message}]

    stream = ollama.chat(model="llama3.2", messages=messages, stream=True)

    acumulado = ""
    for chunk in stream:
        acumulado += chunk["message"]["content"]
        yield acumulado

gr.ChatInterface(chat_streaming).launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
